**Import the feature engineering module**


In [18]:
%run /feature_engineering

StatementMeta(, e52c4f04-325b-4091-956a-63cb5d8554d3, 3, Finished, Available, Finished, True)

In [23]:
# ===================================================================
# Notebook: FraudDetection_BatchInference_With_MLflow_Model         =
# Purpose: Load MLflow registered model and perform batch inference =
# Environment: Microsoft Fabric (PySpark)                           =
# ===================================================================


# ------------------------------ 
# 1. Import required libraries -
# ------------------------------
import mlflow
import mlflow.spark
from pyspark.sql import functions as F

# NOTE:
# MLFlowTransformer import is not needed here since we are using mlflow.spark
# from synapse.ml.predict import MLFlowTransformer


## -----------------------------------
## 2. Register model (Run ONLY once) -
## -----------------------------------
## NOTE: Skip this step if your model is already registered in MLflow

#mlflow.register_model(
#    "runs:/807afb0d-578d-4f44-a8ad-09bdf6f87f2e/model",
#    "GBTClassifier_FraudDetection_Model"
#)


# ------------------------------------------------------------
# 3. Load model from MLflow Model Registry
# ------------------------------------------------------------
MODEL_NAME = "GBTClassifier_FraudDetection_Model"
MODEL_VERSION = "1"   # You can change to 'Production' if promoted

model = mlflow.spark.load_model(
    model_uri=f"models:/{MODEL_NAME}/{MODEL_VERSION}"
)

print(f"✅ Loaded model: {MODEL_NAME} (version {MODEL_VERSION})")


# ------------------------------------------------------------
# 4. Load input data from Lakehouse table or spark DataFrame
# ------------------------------------------------------------


#df = spark.read.table(
#    "FraudDetection.dbo.silver_data_for_mlmodel"
#)

input_json = [{
    "step": 378,
    "type": 'PAYMENT',
    "amount": 75096.27,
    "nameOrig": 'C2028658958',
    "oldbalanceOrg": 3469.15,
    "newbalanceOrig": 293967.22,
    "nameDest": 'M456706057',
    "oldbalanceDest": 532061.26,
    "newbalanceDest": 552286.41,    
}]

df_init = transform_fraudData_features(input_json)

# Drop the string value columns
df_ = df_init.drop('nameOrig')
df_filtered = df_.drop('nameDest') 
print("✅ Input data loaded successfully")


# ------------------------------------------------------------
# 5. Perform predictions using the loaded model
# ------------------------------------------------------------
# Spark ML models use `.transform()` for inference

preds = model.transform(df_filtered)

print("✅ Predictions generated")
display(preds)

StatementMeta(, e52c4f04-325b-4091-956a-63cb5d8554d3, 4, Finished, Available, Finished, False)

2026/06/08 14:25:54 INFO mlflow.spark: 'models:/GBTClassifier_FraudDetection_Model/1' resolved as 'abfss://f8e3d19d-f7c2-4966-9804-d462790c6dac@onelakenortheurope.pbidedicated.windows.net/7e9638a8-12c5-4dda-8386-755064d0dc2f/Data/33bf2fd1-900a-4f10-a4d7-82d2ff19cd16/artifacts'


2026/06/08 14:25:57 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false


2026/06/08 14:25:58 INFO mlflow.spark: File 'models:/GBTClassifier_FraudDetection_Model/1/sparkml' not found on DFS. Will attempt to upload the file.


2026/06/08 14:26:01 INFO mlflow.spark: Copied SparkML model to Files/tmp/mlflow/b93fa068-774d-4894-b5a6-fdf512064a3d


StatementMeta(, e52c4f04-325b-4091-956a-63cb5d8554d3, 5, Finished, Available, Finished, False)

✅ Input data loaded successfully
✅ Predictions generated


SynapseWidget(Synapse.DataFrame, 4fed95f0-3e05-40e7-81fa-b74a88ac5dfb)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [29]:
# ------------------------------------------------------------
# 6. Save prediction results to Lakehouse table
# ------------------------------------------------------------
OUTPUT_TABLE = "FraudDetection.dbo.FraudDetectionModel_Output"

preds.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

print(f"✅ Predictions saved to table: {OUTPUT_TABLE}")


# ------------------------------------------------------------
# 7. Extract sample prediction (for quick debugging / API use)
# ------------------------------------------------------------
# WARNING:
# `.collect()` should only be used for small datasets

if preds.count() > 0:
    result = preds.select("prediction").limit(1).collect()[0][0]
    print('\n')
    print( {"isFraud_prediction": float(result)})
else:
    print("⚠️ No records found for given filter condition")

StatementMeta(, e52c4f04-325b-4091-956a-63cb5d8554d3, 6, Finished, Available, Finished, False)

✅ Predictions saved to table: FraudDetection.dbo.FraudDetectionModel_Output




{'isFraud_prediction': 0.0}
